In [1]:
pip install -U torch transformers accelerate sentencepiece tqdm numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 172.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 180.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 255.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 303.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 290.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 311.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 108.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 184.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 273.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 234.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.

In [3]:
pip install python-dotenv


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from dotenv import load_dotenv
import os

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")

In [6]:
from huggingface_hub import login
login()

In [11]:
pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2
Note: you may need to restart the kernel to use updated packages.


In [1]:
pip uninstall -y torchvision && pip install -U torch transformers accelerate sentencepiece tqdm numpy

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
import json
import random
import importlib.util
from typing import List, Dict, Any, Tuple

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN", "")
HF_HOME = os.getenv("HF_HOME", "")
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUT_DIR = "results_steering"

DEV_RATIO = 0.2
SEED = 42
MAX_LEN = 512
MAX_STEER_EXAMPLES = 2400
ALPHAS = [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]
LAYER_FRACTION = 0.25
NORMALIZE_VECTORS = False
USE_BFLOAT16 = True
USE_CHAT_TEMPLATE = True
YES_TEXT = " yes"
NO_TEXT = " no"
RUN_TEST = False

if HF_HOME:
    os.environ["HF_HOME"] = HF_HOME


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def stratified_split(items, dev_ratio, seed):
    rng = random.Random(seed)
    buckets = {}
    for it in items:
        key = (it["validity"], it["plausibility"])
        buckets.setdefault(key, []).append(it)
    train, dev = [], []
    for bucket in buckets.values():
        rng.shuffle(bucket)
        n_dev = max(1, int(len(bucket) * dev_ratio)) if len(bucket) > 5 else max(0, int(len(bucket) * dev_ratio))
        dev.extend(bucket[:n_dev])
        train.extend(bucket[n_dev:])
    rng.shuffle(train)
    rng.shuffle(dev)
    return train, dev


def run_official_eval(eval_script_path, ref_path, pred_path, out_path):
    spec = importlib.util.spec_from_file_location("semeval_eval", eval_script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.run_full_scoring(ref_path, pred_path, out_path)
    result = load_json(out_path)
    return result


def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "module"):
        return get_layers(model.module)
    raise ValueError("Unsupported model architecture")


def get_target_layers(total, fraction):
    start = int(total * (1.0 - fraction))
    return list(range(start, total))


def build_prompt(tokenizer, syllogism):
    messages = [
        {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
        {"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."}
    ]
    if USE_CHAT_TEMPLATE:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        "You are a strict formal logic reasoner.\n"
        "Decide only whether the conclusion logically follows from the premises.\n"
        "Ignore plausibility and world knowledge.\n"
        "Reply with only yes or no.\n\n"
        f"Argument:\n{syllogism}\n\nAnswer:"
    )


def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    return y >= n


def get_all_layer_hidden(model, tokenizer, text, target_layers, max_len):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    return {l: out.hidden_states[l + 1][0, -1, :].float().cpu() for l in target_layers}


def balanced_by_validity(data, n, seed):
    rng = random.Random(seed)
    pos = [x for x in data if x["validity"] is True]
    neg = [x for x in data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = n // 2
    out = pos[:half] + neg[:n - half]
    rng.shuffle(out)
    return out


def compute_steering_vectors(model, tokenizer, train_items, target_layers):
    plausible = [x for x in train_items if x["plausibility"] is True]
    implausible = [x for x in train_items if x["plausibility"] is False]
    n = min(len(plausible), len(implausible), MAX_STEER_EXAMPLES // 2)
    plausible_sample = balanced_by_validity(plausible, n, SEED)
    implausible_sample = balanced_by_validity(implausible, n, SEED)
    print(f"Steering: {n} plausible + {n} implausible")

    plausible_acts = {l: [] for l in target_layers}
    implausible_acts = {l: [] for l in target_layers}

    print("Extracting plausible activations...")
    for i, ex in enumerate(plausible_sample):
        prompt = build_prompt(tokenizer, ex["syllogism"])
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers, MAX_LEN)
        for l in target_layers:
            plausible_acts[l].append(hiddens[l])
        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{n}")

    print("Extracting implausible activations...")
    for i, ex in enumerate(implausible_sample):
        prompt = build_prompt(tokenizer, ex["syllogism"])
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers, MAX_LEN)
        for l in target_layers:
            implausible_acts[l].append(hiddens[l])
        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{n}")

    deltas = {}
    for l in target_layers:
        mu_p = torch.stack(plausible_acts[l]).mean(dim=0)
        mu_ip = torch.stack(implausible_acts[l]).mean(dim=0)
        delta = mu_ip - mu_p
        if NORMALIZE_VECTORS:
            delta = delta / delta.norm()
        deltas[l] = delta
        print(f"  Layer {l}: ||delta|| = {delta.norm():.4f}")

    return deltas


class StaticHook:
    def __init__(self, delta, alpha):
        self.delta = delta
        self.alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            d = self.delta.to(x.device, x.dtype)
            return (x + self.alpha * d.view(1, 1, -1),) + output[1:]
        d = self.delta.to(output.device, output.dtype)
        return output + self.alpha * d.view(1, 1, -1)


def register_hooks(layers, deltas, alpha):
    handles = []
    for layer_idx, delta in deltas.items():
        handles.append(layers[layer_idx].register_forward_hook(StaticHook(delta, alpha)))
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


@torch.no_grad()
def predict_dataset(model, tokenizer, eval_items, deltas, alpha):
    layers = get_layers(model)
    handles = []
    if deltas is not None and alpha != 0:
        handles = register_hooks(layers, deltas, alpha)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


def print_subgroup_accuracy(dev_items, preds):
    pred_map = {p["id"]: p["validity"] for p in preds}
    buckets = {}
    for item in dev_items:
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item["plausibility"] else "implausible"
        key = f"{v}_{p}"
        buckets.setdefault(key, {"correct": 0, "total": 0})
        buckets[key]["total"] += 1
        if pred_map.get(item["id"]) == item["validity"]:
            buckets[key]["correct"] += 1
    for key in sorted(buckets.keys()):
        vals = buckets[key]
        acc = 100 * vals["correct"] / max(1, vals["total"])
        print(f"  {key:25s}: {acc:6.2f}%  ({vals['correct']}/{vals['total']})")


def main():
    set_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Model: {MODEL_NAME}")
    print(f"Alphas: {ALPHAS}")
    os.makedirs(OUT_DIR, exist_ok=True)

    token = HF_TOKEN if HF_TOKEN else None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if USE_BFLOAT16 and device == "cuda" else (torch.float16 if device == "cuda" else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=token, torch_dtype=dtype, device_map="auto" if device == "cuda" else None)
    if device != "cuda":
        model.to(device)
    model.eval()
    model.config.use_cache = False

    items = load_json(TRAIN_JSON)
    train_items, dev_items = stratified_split(items, DEV_RATIO, SEED)
    save_json(train_items, os.path.join(OUT_DIR, "train_split.json"))
    save_json(dev_items, os.path.join(OUT_DIR, "dev_split.json"))
    print(f"Train: {len(train_items)}, Dev: {len(dev_items)}")

    all_layers = get_layers(model)
    target_layers = get_target_layers(len(all_layers), LAYER_FRACTION)
    print(f"Total layers: {len(all_layers)}, Steering layers: {target_layers}")

    deltas = compute_steering_vectors(model, tokenizer, train_items, target_layers)
    torch.save(deltas, os.path.join(OUT_DIR, "steering_vectors.pt"))

    best_score = -1
    best_alpha = 0
    summary = []

    for alpha in ALPHAS:
        print(f"\n===== Alpha = {alpha} =====")
        preds = predict_dataset(model, tokenizer, dev_items, deltas, alpha)
        tag = str(alpha).replace("-", "neg").replace(".", "p")
        pred_path = os.path.join(OUT_DIR, f"preds_alpha_{tag}.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(dev_items, preds)
        result = run_official_eval(EVAL_SCRIPT, os.path.join(OUT_DIR, "dev_split.json"), pred_path, os.path.join(OUT_DIR, f"eval_alpha_{tag}.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        summary.append({"alpha": alpha, "accuracy": result["accuracy"], "content_effect": result["content_effect"], "combined_score": result["combined_score"]})
        if result["combined_score"] > best_score:
            best_score = result["combined_score"]
            best_alpha = alpha

    print("\n===== SWEEP SUMMARY =====")
    print(f"{'Alpha':>8s}  {'ACC':>8s}  {'TCE':>10s}  {'Score':>10s}")
    for s in summary:
        marker = " <-- best" if s["alpha"] == best_alpha else ""
        print(f"{s['alpha']:>8.1f}  {s['accuracy']:>8.2f}  {s['content_effect']:>10.4f}  {s['combined_score']:>10.4f}{marker}")
    print(f"\nBest alpha: {best_alpha} (combined_score: {best_score:.4f})")

    save_json(summary, os.path.join(OUT_DIR, "sweep_summary.json"))

    if RUN_TEST and TEST_JSON:
        print(f"\n===== Test predictions with alpha={best_alpha} =====")
        test_items = load_json(TEST_JSON)
        test_preds = predict_dataset(model, tokenizer, test_items, deltas, best_alpha)
        save_json(test_preds, os.path.join(OUT_DIR, "predictions_subtask1.json"))
        print(f"Saved {len(test_preds)} test predictions")


if __name__ == "__main__":
    main()

Device: cuda
Model: Qwen/Qwen2.5-1.5B-Instruct
Alphas: [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Train: 769, Dev: 191
Total layers: 28, Steering layers: [21, 22, 23, 24, 25, 26, 27]
Steering: 380 plausible + 380 implausible
Extracting plausible activations...
  100/380
  200/380
  300/380
Extracting implausible activations...
  100/380
  200/380
  300/380
  Layer 21: ||delta|| = 9.7492
  Layer 22: ||delta|| = 14.5339
  Layer 23: ||delta|| = 14.8219
  Layer 24: ||delta|| = 16.2271
  Layer 25: ||delta|| = 19.0641
  Layer 26: ||delta|| = 19.4499
  Layer 27: ||delta|| = 18.7419

===== Alpha = 0 =====
  50/191
  100/191
  150/191
  invalid_implausible      :  61.22%  (30/49)
  invalid_plausible        :  10.87%  (5/46)
  valid_implausible        :  83.33%  (40/48)
  valid_plausible          : 100.00%  (48/48)
Scoring complete. Results written to results_steering/eval_alpha_0.json
  ACC=64.40  TCE=44.5652  Score=13.3629

===== Alpha = -5 =====
  50/191
  100/191
  150/191
  invalid_implausible      :   0.00%  (0/49)
  invalid_plausible        :   0.00%  (0/46)
  valid_implausible       

In [9]:
import os
import json
import random
import importlib.util
from typing import List, Dict, Any, Tuple

from dotenv import load_dotenv
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

load_dotenv()

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN")
HF_HOME = os.getenv("HF_HOME")

if HF_HOME:
    os.environ["HF_HOME"] = HF_HOME

TRAIN_JSON = "train_data/subtask 1/train_data.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUT_DIR = "results_llama32_1b_static"

DEV_RATIO = 0.2
SEED = 42
MAX_LEN = 512
MAX_STEER_EXAMPLES = 2400
ALPHA = 0.0
PROMPT_MODE = "zero_shot"
ICL_SHOTS = 4
USE_BFLOAT16 = True
USE_CHAT_TEMPLATE = True
TARGET_LAYER_MODE = "last_quarter"

YES_TEXT = " yes"
NO_TEXT = " no"


def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path: str):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def stratified_split(items: List[Dict[str, Any]], dev_ratio: float, seed: int) -> Tuple[List, List]:
    rng = random.Random(seed)
    buckets = {}
    for it in items:
        key = (it["validity"], it["plausibility"])
        buckets.setdefault(key, []).append(it)
    train, dev = [], []
    for bucket in buckets.values():
        rng.shuffle(bucket)
        n_dev = max(1, int(len(bucket) * dev_ratio)) if len(bucket) > 5 else max(0, int(len(bucket) * dev_ratio))
        dev.extend(bucket[:n_dev])
        train.extend(bucket[n_dev:])
    rng.shuffle(train)
    rng.shuffle(dev)
    return train, dev


def run_official_eval(eval_script_path: str, ref_path: str, pred_path: str, out_path: str):
    spec = importlib.util.spec_from_file_location("semeval_eval", eval_script_path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    module.run_full_scoring(ref_path, pred_path, out_path)
    with open(out_path, "r", encoding="utf-8") as f:
        print(json.load(f))


def get_layers(model):
    base = model
    if hasattr(base, "model") and hasattr(base.model, "layers"):
        return base.model.layers
    if hasattr(base, "transformer") and hasattr(base.transformer, "h"):
        return base.transformer.h
    if hasattr(base, "module"):
        return get_layers(base.module)
    raise ValueError("Unsupported model architecture")


def choose_target_layers(n: int):
    if TARGET_LAYER_MODE == "last_quarter":
        s = int(n * 0.75)
        return list(range(s, n))
    if TARGET_LAYER_MODE == "last_third":
        s = int(n * (2.0 / 3.0))
        return list(range(s, n))
    if TARGET_LAYER_MODE == "last_half":
        s = int(n * 0.5)
        return list(range(s, n))
    return [n - 1]


def build_zero_shot_messages(syllogism: str):
    return [
        {
            "role": "system",
            "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."
        },
        {
            "role": "user",
            "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."
        }
    ]


def build_plain_zero_shot_prompt(syllogism: str):
    return (
        "You are a strict formal logic reasoner.\n"
        "Decide only whether the conclusion logically follows from the premises.\n"
        "Ignore plausibility and world knowledge.\n"
        "Reply with only yes or no.\n\n"
        f"Argument:\n{syllogism}\n\n"
        "Answer:"
    )


def build_icl_examples(train_data: List[Dict[str, Any]], shots: int, seed: int):
    rng = random.Random(seed)
    pos = [x for x in train_data if x["validity"] is True]
    neg = [x for x in train_data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    chosen = []
    half = shots // 2
    chosen.extend(pos[:half])
    chosen.extend(neg[:shots - half])
    rng.shuffle(chosen)
    return chosen


def build_icl_messages(train_data: List[Dict[str, Any]], shots: int, seed: int, syllogism: str):
    chosen = build_icl_examples(train_data, shots, seed)
    messages = [
        {
            "role": "system",
            "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."
        }
    ]
    for ex in chosen:
        ans = "yes" if ex["validity"] else "no"
        messages.append({"role": "user", "content": f"Argument:\n{ex['syllogism']}\n\nAnswer yes or no."})
        messages.append({"role": "assistant", "content": ans})
    messages.append({"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."})
    return messages


def build_plain_icl_prompt(train_data: List[Dict[str, Any]], shots: int, seed: int, syllogism: str):
    chosen = build_icl_examples(train_data, shots, seed)
    parts = [
        "You are a strict formal logic reasoner.",
        "Decide only whether the conclusion logically follows from the premises.",
        "Ignore plausibility and world knowledge.",
        "Reply with only yes or no.",
        ""
    ]
    for ex in chosen:
        ans = "yes" if ex["validity"] else "no"
        parts.append("Argument:")
        parts.append(ex["syllogism"])
        parts.append(f"Answer: {ans}")
        parts.append("")
    parts.append("Argument:")
    parts.append(syllogism)
    parts.append("Answer:")
    return "\n".join(parts)


def build_prompt(tokenizer, train_data, syllogism: str):
    if PROMPT_MODE == "icl":
        if USE_CHAT_TEMPLATE:
            messages = build_icl_messages(train_data, ICL_SHOTS, SEED, syllogism)
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        return build_plain_icl_prompt(train_data, ICL_SHOTS, SEED, syllogism)
    if USE_CHAT_TEMPLATE:
        messages = build_zero_shot_messages(syllogism)
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return build_plain_zero_shot_prompt(syllogism)


def score_label(model, tokenizer, prompt: str, label_text: str, max_len: int):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity(model, tokenizer, prompt: str, max_len: int):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    return y >= n


def get_hidden_last_token(model, tokenizer, text: str, layer_idx: int, max_len: int):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    return out.hidden_states[layer_idx + 1][0, -1, :].float().cpu()


def balanced_sample_by_validity(data: List[Dict[str, Any]], n: int, seed: int):
    rng = random.Random(seed)
    if len(data) <= n:
        out = data[:]
        rng.shuffle(out)
        return out
    pos = [x for x in data if x["validity"] is True]
    neg = [x for x in data if x["validity"] is False]
    half = n // 2
    rng.shuffle(pos)
    rng.shuffle(neg)
    out = pos[:half] + neg[:n - half]
    rng.shuffle(out)
    return out


def mean_stack(vecs):
    return torch.stack(vecs, dim=0).mean(dim=0)


def collect_training_states(model, tokenizer, train_items: List[Dict[str, Any]], probe_layer: int, max_len: int):
    steering_pool = balanced_sample_by_validity(train_items, MAX_STEER_EXAMPLES, SEED)
    correct_vecs = []
    wrong_vecs = []
    for ex in steering_pool:
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        h = get_hidden_last_token(model, tokenizer, prompt, probe_layer, max_len)
        pred = predict_validity(model, tokenizer, prompt, max_len)
        gold = bool(ex["validity"])
        if pred == gold:
            correct_vecs.append(h)
        else:
            wrong_vecs.append(h)
    return correct_vecs, wrong_vecs


class StaticHook:
    def __init__(self, delta, alpha):
        self.delta = delta
        self.alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        d = self.delta.to(x.device, x.dtype)
        x = x + self.alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


def register_static_hooks(layers, layer_ids, delta, alpha):
    handles = []
    hook = StaticHook(delta, alpha)
    for i in layer_ids:
        handles.append(layers[i].register_forward_hook(hook))
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


@torch.no_grad()
def predict_to_json_static(model, tokenizer, train_items: List[Dict[str, Any]], dev_items: List[Dict[str, Any]], layer_ids, delta, alpha, max_len: int, out_path: str):
    layers = get_layers(model)
    handles = register_static_hooks(layers, layer_ids, delta, alpha)
    preds = []
    try:
        for ex in dev_items:
            prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, max_len)
            preds.append({"id": ex["id"], "validity": bool(pred)})
    finally:
        remove_hooks(handles)
    save_json(preds, out_path)
    print("Wrote predictions:", out_path)


def main():
    if not HF_TOKEN:
        raise ValueError("HF_TOKEN not found. Put it in .env or export it before running.")

    set_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    os.makedirs(OUT_DIR, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if USE_BFLOAT16 and device == "cuda" else (torch.float16 if device == "cuda" else torch.float32)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        torch_dtype=dtype,
        device_map="auto" if device == "cuda" else None
    )

    if device != "cuda":
        model.to(device)

    model.eval()
    model.config.use_cache = False

    items = load_json(TRAIN_JSON)
    train_items, dev_items = stratified_split(items, DEV_RATIO, SEED)

    train_split_path = os.path.join(OUT_DIR, "train_split.json")
    dev_split_path = os.path.join(OUT_DIR, "dev_split.json")
    save_json(train_items, train_split_path)
    save_json(dev_items, dev_split_path)

    layers = get_layers(model)
    layer_ids = choose_target_layers(len(layers))
    probe_layer = layer_ids[0]

    correct_vecs, wrong_vecs = collect_training_states(model, tokenizer, train_items, probe_layer, MAX_LEN)

    if len(correct_vecs) == 0 or len(wrong_vecs) == 0:
        raise ValueError("Need both correct and wrong prediction activations to build steering vector")

    delta = mean_stack(correct_vecs) - mean_stack(wrong_vecs)

    pred_path = os.path.join(OUT_DIR, "dev_predictions.json")
    predict_to_json_static(
        model=model,
        tokenizer=tokenizer,
        train_items=train_items,
        dev_items=dev_items,
        layer_ids=layer_ids,
        delta=delta,
        alpha=ALPHA,
        max_len=MAX_LEN,
        out_path=pred_path
    )

    eval_out = os.path.join(OUT_DIR, "official_eval_dev.json")
    run_official_eval(EVAL_SCRIPT, dev_split_path, pred_path, eval_out)


if __name__ == "__main__":
    main()

Device: cuda


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Wrote predictions: results_llama32_1b_static/dev_predictions.json
Scoring complete. Results written to results_llama32_1b_static/official_eval_dev.json
{'accuracy': 50.2618, 'content_effect': 50.0, 'combined_score': 10.1913}


In [10]:
import os
import json
import random
import importlib.util
from typing import List, Dict, Any, Tuple

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN", "")
HF_HOME = os.getenv("HF_HOME", "")
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUT_DIR = "results_steering"

DEV_RATIO = 0.2
SEED = 42
MAX_LEN = 512
MAX_STEER_EXAMPLES = 2400
ALPHAS = [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]
LAYER_FRACTION = 0.5
NORMALIZE_VECTORS = False
USE_BFLOAT16 = True
USE_CHAT_TEMPLATE = True
YES_TEXT = " yes"
NO_TEXT = " no"
RUN_TEST = False
MODE = "kcast"

if HF_HOME:
    os.environ["HF_HOME"] = HF_HOME


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def stratified_split(items, dev_ratio, seed):
    rng = random.Random(seed)
    buckets = {}
    for it in items:
        key = (it["validity"], it["plausibility"])
        buckets.setdefault(key, []).append(it)
    train, dev = [], []
    for bucket in buckets.values():
        rng.shuffle(bucket)
        n_dev = max(1, int(len(bucket) * dev_ratio)) if len(bucket) > 5 else max(0, int(len(bucket) * dev_ratio))
        dev.extend(bucket[:n_dev])
        train.extend(bucket[n_dev:])
    rng.shuffle(train)
    rng.shuffle(dev)
    return train, dev


def run_official_eval(eval_script_path, ref_path, pred_path, out_path):
    spec = importlib.util.spec_from_file_location("semeval_eval", eval_script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.run_full_scoring(ref_path, pred_path, out_path)
    return load_json(out_path)


def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "module"):
        return get_layers(model.module)
    raise ValueError("Unsupported model architecture")


def get_target_layers(total, fraction):
    start = int(total * (1.0 - fraction))
    return list(range(start, total))


def build_prompt(tokenizer, syllogism):
    messages = [
        {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
        {"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."}
    ]
    if USE_CHAT_TEMPLATE:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        "You are a strict formal logic reasoner.\n"
        "Decide only whether the conclusion logically follows from the premises.\n"
        "Ignore plausibility and world knowledge.\n"
        "Reply with only yes or no.\n\n"
        f"Argument:\n{syllogism}\n\nAnswer:"
    )


def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    return y >= n


def get_all_layer_hidden(model, tokenizer, text, target_layers, max_len):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    return {l: out.hidden_states[l + 1][0, -1, :].float().cpu() for l in target_layers}


def balanced_by_validity(data, n, seed):
    rng = random.Random(seed)
    pos = [x for x in data if x["validity"] is True]
    neg = [x for x in data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = n // 2
    out = pos[:half] + neg[:n - half]
    rng.shuffle(out)
    return out


def compute_steering_data(model, tokenizer, train_items, target_layers):
    plausible = [x for x in train_items if x["plausibility"] is True]
    implausible = [x for x in train_items if x["plausibility"] is False]
    n = min(len(plausible), len(implausible), MAX_STEER_EXAMPLES // 2)
    pool = balanced_by_validity(plausible, n, SEED) + balanced_by_validity(implausible, n, SEED)
    random.Random(SEED).shuffle(pool)
    print(f"Steering pool: {len(pool)} examples")

    correct_acts = {l: [] for l in target_layers}
    wrong_acts = {l: [] for l in target_layers}
    valid_acts = {l: [] for l in target_layers}
    invalid_acts = {l: [] for l in target_layers}

    n_correct = 0
    n_wrong = 0

    for i, ex in enumerate(pool):
        prompt = build_prompt(tokenizer, ex["syllogism"])
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers, MAX_LEN)
        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        gold = bool(ex["validity"])
        is_correct = (pred == gold)

        if is_correct:
            n_correct += 1
        else:
            n_wrong += 1

        for l in target_layers:
            if is_correct:
                correct_acts[l].append(hiddens[l])
            else:
                wrong_acts[l].append(hiddens[l])
            if gold:
                valid_acts[l].append(hiddens[l])
            else:
                invalid_acts[l].append(hiddens[l])

        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(pool)} (correct={n_correct}, wrong={n_wrong})")

    print(f"  Total correct={n_correct}, wrong={n_wrong}")

    deltas = {}
    for l in target_layers:
        if len(correct_acts[l]) == 0 or len(wrong_acts[l]) == 0:
            raise ValueError(f"Layer {l}: need both correct and wrong examples")
        mu_plus = torch.stack(correct_acts[l]).mean(dim=0)
        mu_minus = torch.stack(wrong_acts[l]).mean(dim=0)
        delta = mu_plus - mu_minus
        if NORMALIZE_VECTORS:
            delta = delta / delta.norm()
        deltas[l] = delta
        print(f"  Layer {l}: ||delta|| = {delta.norm():.4f}")

    condition_valid = {}
    condition_invalid = {}
    for l in target_layers:
        condition_valid[l] = torch.stack(valid_acts[l]).mean(dim=0)
        condition_invalid[l] = torch.stack(invalid_acts[l]).mean(dim=0)

    return deltas, condition_valid, condition_invalid


def project_onto(v, u):
    dot = torch.dot(v, u)
    return dot / (torch.dot(u, u) + 1e-10) * u


def cosine_sim(a, b):
    return torch.dot(a, b) / (a.norm() * b.norm() + 1e-10)


class StaticHook:
    def __init__(self, delta, alpha):
        self.delta = delta
        self.alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            d = self.delta.to(x.device, x.dtype)
            return (x + self.alpha * d.view(1, 1, -1),) + output[1:]
        d = self.delta.to(output.device, output.dtype)
        return output + self.alpha * d.view(1, 1, -1)


class KCASTHook:
    def __init__(self, delta, alpha, psi_valid, psi_invalid, layer_idx):
        self.delta = delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid
        self.layer_idx = layer_idx
        self.effective_alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None

        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()

        if sim_valid > sim_invalid:
            effective_alpha = -self.alpha
        else:
            effective_alpha = self.alpha

        self.effective_alpha = effective_alpha
        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)

        if rest is None:
            return x
        return (x,) + rest


def register_static_hooks(layers, deltas, alpha):
    handles = []
    for layer_idx, delta in deltas.items():
        handles.append(layers[layer_idx].register_forward_hook(StaticHook(delta, alpha)))
    return handles


def register_kcast_hooks(layers, deltas, alpha, cond_valid, cond_invalid):
    handles = []
    for layer_idx, delta in deltas.items():
        hook = KCASTHook(delta, alpha, cond_valid[layer_idx], cond_invalid[layer_idx], layer_idx)
        handles.append(layers[layer_idx].register_forward_hook(hook))
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


@torch.no_grad()
def predict_dataset_static(model, tokenizer, eval_items, deltas, alpha):
    layers = get_layers(model)
    handles = []
    if deltas is not None and alpha != 0:
        handles = register_static_hooks(layers, deltas, alpha)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


@torch.no_grad()
def predict_dataset_kcast(model, tokenizer, eval_items, deltas, alpha, cond_valid, cond_invalid):
    layers = get_layers(model)
    handles = register_kcast_hooks(layers, deltas, alpha, cond_valid, cond_invalid)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


def print_subgroup_accuracy(dev_items, preds):
    pred_map = {p["id"]: p["validity"] for p in preds}
    buckets = {}
    for item in dev_items:
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item["plausibility"] else "implausible"
        key = f"{v}_{p}"
        buckets.setdefault(key, {"correct": 0, "total": 0})
        buckets[key]["total"] += 1
        if pred_map.get(item["id"]) == item["validity"]:
            buckets[key]["correct"] += 1
    for key in sorted(buckets.keys()):
        vals = buckets[key]
        acc = 100 * vals["correct"] / max(1, vals["total"])
        print(f"  {key:25s}: {acc:6.2f}%  ({vals['correct']}/{vals['total']})")


def main():
    set_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Model: {MODEL_NAME}")
    print(f"Mode: {MODE}")
    print(f"Alphas: {ALPHAS}")
    os.makedirs(OUT_DIR, exist_ok=True)

    token = HF_TOKEN if HF_TOKEN else None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if USE_BFLOAT16 and device == "cuda" else (torch.float16 if device == "cuda" else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=token, torch_dtype=dtype, device_map="auto" if device == "cuda" else None)
    if device != "cuda":
        model.to(device)
    model.eval()
    model.config.use_cache = False

    items = load_json(TRAIN_JSON)
    train_items, dev_items = stratified_split(items, DEV_RATIO, SEED)
    save_json(train_items, os.path.join(OUT_DIR, "train_split.json"))
    save_json(dev_items, os.path.join(OUT_DIR, "dev_split.json"))
    print(f"Train: {len(train_items)}, Dev: {len(dev_items)}")

    all_layers = get_layers(model)
    target_layers = get_target_layers(len(all_layers), LAYER_FRACTION)
    print(f"Total layers: {len(all_layers)}, Steering layers: {target_layers}")

    deltas, cond_valid, cond_invalid = compute_steering_data(model, tokenizer, train_items, target_layers)
    torch.save({"deltas": deltas, "cond_valid": cond_valid, "cond_invalid": cond_invalid}, os.path.join(OUT_DIR, "steering_data.pt"))

    best_score = -1
    best_alpha = 0
    summary = []

    for alpha in ALPHAS:
        print(f"\n===== {MODE.upper()} Alpha = {alpha} =====")
        if MODE == "kcast":
            if alpha == 0:
                preds = predict_dataset_static(model, tokenizer, dev_items, deltas, 0)
            else:
                preds = predict_dataset_kcast(model, tokenizer, dev_items, deltas, alpha, cond_valid, cond_invalid)
        else:
            preds = predict_dataset_static(model, tokenizer, dev_items, deltas, alpha)

        tag = str(alpha).replace("-", "neg").replace(".", "p")
        pred_path = os.path.join(OUT_DIR, f"preds_{MODE}_alpha_{tag}.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(dev_items, preds)
        result = run_official_eval(EVAL_SCRIPT, os.path.join(OUT_DIR, "dev_split.json"), pred_path, os.path.join(OUT_DIR, f"eval_{MODE}_alpha_{tag}.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        summary.append({"alpha": alpha, "accuracy": result["accuracy"], "content_effect": result["content_effect"], "combined_score": result["combined_score"]})
        if result["combined_score"] > best_score:
            best_score = result["combined_score"]
            best_alpha = alpha

    print(f"\n===== SWEEP SUMMARY ({MODE.upper()}) =====")
    print(f"{'Alpha':>8s}  {'ACC':>8s}  {'TCE':>10s}  {'Score':>10s}")
    for s in summary:
        marker = " <-- best" if s["alpha"] == best_alpha else ""
        print(f"{s['alpha']:>8.1f}  {s['accuracy']:>8.2f}  {s['content_effect']:>10.4f}  {s['combined_score']:>10.4f}{marker}")
    print(f"\nBest alpha: {best_alpha} (combined_score: {best_score:.4f})")
    save_json(summary, os.path.join(OUT_DIR, f"sweep_summary_{MODE}.json"))

    if RUN_TEST and TEST_JSON:
        print(f"\n===== Test predictions with alpha={best_alpha} =====")
        test_items = load_json(TEST_JSON)
        if MODE == "kcast":
            test_preds = predict_dataset_kcast(model, tokenizer, test_items, deltas, best_alpha, cond_valid, cond_invalid)
        else:
            test_preds = predict_dataset_static(model, tokenizer, test_items, deltas, best_alpha)
        save_json(test_preds, os.path.join(OUT_DIR, "predictions_subtask1.json"))
        print(f"Saved {len(test_preds)} test predictions")


if __name__ == "__main__":
    main()

Device: cuda
Model: meta-llama/Llama-3.2-1B-Instruct
Mode: kcast
Alphas: [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Train: 769, Dev: 191
Total layers: 16, Steering layers: [8, 9, 10, 11, 12, 13, 14, 15]
Steering pool: 758 examples
  100/758 (correct=44, wrong=56)
  200/758 (correct=97, wrong=103)
  300/758 (correct=145, wrong=155)
  400/758 (correct=194, wrong=206)
  500/758 (correct=250, wrong=250)
  600/758 (correct=304, wrong=296)
  700/758 (correct=353, wrong=347)
  Total correct=380, wrong=378
  Layer 8: ||delta|| = 0.1741
  Layer 9: ||delta|| = 0.2072
  Layer 10: ||delta|| = 0.2332
  Layer 11: ||delta|| = 0.2640
  Layer 12: ||delta|| = 0.3230
  Layer 13: ||delta|| = 0.3981
  Layer 14: ||delta|| = 0.4877
  Layer 15: ||delta|| = 3.7656

===== KCAST Alpha = 0 =====
  50/191
  100/191
  150/191
  invalid_implausible      :   0.00%  (0/49)
  invalid_plausible        :   0.00%  (0/46)
  valid_implausible        : 100.00%  (48/48)
  valid_plausible          : 100.00%  (48/48)
Scoring complete. Results written to results_steering/eval_kcast_alpha_0.json
  ACC=50.26  TCE=50.0000  Score=10.1913

===== K